In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error

## Preparacion de datos

In [4]:
# df = pd.read_csv('data/cleaned.csv')
# X_train, X_test = train_test_split(df, test_size=0.2, stratify=df['Modelo'], random_state=42)
# # Save X_test to a CSV file inside the data folder
# X_test.to_csv('data/cleaned_test.csv', index=True)
# print(f"X_test dataset with {X_test.shape[0]} rows and {X_test.shape[1]} columns saved to data/cleaned_test.csv")
# X_train.to_csv('data/cleaned_train.csv', index=True)
# print(f"X_train dataset with {X_train.shape[0]} rows and {X_train.shape[1]} columns saved to data/cleaned_train.csv")

In [5]:
df = pd.read_csv('../data/train/cleaned_train.csv')

# Convert the 'Moneda' column to binary (1 for 'US$', 0 for '$')
df['Moneda'] = df['Moneda'].apply(lambda x: 1 if x == 'US$' else 0)
df.loc[df['Moneda'] == 1, 'Precio'] = df.loc[df['Moneda'] == 1, 'Precio'] * 1100

# Convert Kilometers column to numeric, removing 'km' if present
df['Kilómetros'] = df['Kilómetros'].str.replace(' km', '', regex=False).str.replace('km', '', regex=False)
df['Kilómetros'] = pd.to_numeric(df['Kilómetros'], errors='coerce')

# Handle the price conversion for US$ values (Moneda = 1)
# Assuming 1 US$ = 1100 local currency

def extraer_formato_correcto(texto):
    if isinstance(texto, str):
        match = re.search(r'\b(\d\.\d)', texto, re.IGNORECASE)
        return match.group(1) if match else np.nan
    return np.nan

# Aplicar solo sobre la columna 'Motor'
df['Motor'] = df['Motor'].apply(extraer_formato_correcto).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14603 entries, 0 to 14602
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   idx                      14603 non-null  int64  
 1   Marca                    14603 non-null  object 
 2   Modelo                   14603 non-null  object 
 3   Año                      14603 non-null  float64
 4   Versión                  14085 non-null  object 
 5   Color                    14288 non-null  object 
 6   Tipo de combustible      14603 non-null  object 
 7   Puertas                  14603 non-null  float64
 8   Transmisión              14590 non-null  object 
 9   Motor                    14410 non-null  float64
 10  Tipo de carrocería       14603 non-null  object 
 11  Con cámara de retroceso  3749 non-null   object 
 12  Kilómetros               14603 non-null  float64
 13  Título                   14603 non-null  object 
 14  Precio                

In [6]:
# Variable target
y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio"])

# Train/val split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Target/mean encoding (usar media de train solamente)
mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means)  # aplicar los mismos valores (NaN si valor nuevo)

# Categóricas para one-hot
one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

# Normalizamos las columnas numéricas
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])  # usar la misma escala

# Ordinal encode antes de KNN imputación
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  # usar misma codificación

# Imputación KNN solo con train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

# Volvemos a DataFrame
X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

# One-hot encoding (sin dummy_na)
X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

# Alineamos columnas (puede faltar alguna en val que está en train)
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 39)
Val: (2921, 39)


In [7]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11682 entries, 12283 to 7270
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Marca                        11682 non-null  float64
 1   Modelo                       11682 non-null  float64
 2   Año                          11682 non-null  float64
 3   Versión                      11682 non-null  float64
 4   Color                        11682 non-null  float64
 5   Puertas                      11682 non-null  float64
 6   Motor                        11682 non-null  float64
 7   Kilómetros                   11682 non-null  float64
 8   Moneda                       11682 non-null  float64
 9   Turbo                        11682 non-null  float64
 10  cv                           11682 non-null  float64
 11  Tipo de combustible_0.0      11682 non-null  float64
 12  Tipo de combustible_1.0      11682 non-null  float64
 13  Tipo de combustibl

In [8]:
X_train.describe()

,Marca,Modelo,Año,Versión,Color,Puertas,Motor,Kilómetros,Moneda,Turbo,...,Con cámara de retroceso_1.0,Tracción_0.0,Tracción_0.2,Tracción_0.4,Tracción_0.6,Tracción_0.8,Tracción_1.0,Tipo de vendedor_0.0,Tipo de vendedor_1.0,Tipo de vendedor_2.0
count,1.168200e+04,1.168200e+04,1.168200e+04,11682.000000,11682.000000,1.168200e+04,11682.000000,1.168200e+04,1.168200e+04,1.168200e+04,...,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000,11682.000000
mean,-1.055292e-16,-9.306030e-17,7.600837e-15,-0.007505,0.000139,-1.216474e-18,0.002966,2.296096e-17,2.554596e-17,-6.325667e-17,...,0.521572,0.489385,0.085431,0.070108,0.061548,0.069680,0.223849,0.465331,0.314159,0.220510
std,1.000043e+00,1.000043e+00,1.000043e+00,0.985699,0.991533,1.000043e+00,0.998655,1.000043e+00,1.000043e+00,1.000043e+00,...,0.499556,0.499909,0.279533,0.255340,0.240342,0.254618,0.416840,0.498818,0.464200,0.414608
min,-1.931649e+00,-1.477319e+00,-6.964258e+00,-1.469835,-6.513514,-1.684704e-02,-1.322764,-6.255174e-01,-7.954272e-01,-7.323228e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,-5.715502e-01,-6.011206e-01,-5.527707e-01,-0.555411,-0.081552,-1.304106e-02,-0.506378,-6.255174e-01,-7.954272e-01,-7.323228e-01,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,-3.931298e-01,-2.822843e-01,-3.214603e-03,-0.235988,0.292320,-1.304106e-02,-0.343100,-6.565198e-02,-7.954272e-01,-7.323228e-01,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.758016e-01,4.313425e-01,9.127122e-01,0.235905,0.684673,-1.304106e-02,0.310009,3.354456e-01,1.257186e+00,1.365518e+00,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
max,5.383539e+00,8.595458e+00,1.095898e+00,30.579042,23.421120,7.641989e+01,7.494209,8.293648e+01,1.257186e+00,1.365518e+00,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Base Line

In [9]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Base Line")
mlflow.set_tracking_uri("http://localhost:5000")

In [10]:
reports = []

## Regresor lineal

In [11]:
# Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict on the validation set
y_pred_train_lr = lr.predict(X_train)
y_pred_val_lr = lr.predict(X_val)

In [12]:
# Define a function to calculate all metrics and format a report
def calculate_metrics(y_train, y_pred_train, y_val, y_pred_val):
    # Calculate train metrics
    mse_train = mean_squared_error(y_train, y_pred_train)
    r2_train = r2_score(y_train, y_pred_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    evs_train = explained_variance_score(y_train, y_pred_train)
    max_err_train = max_error(y_train, y_pred_train)
    medae_train = median_absolute_error(y_train, y_pred_train)
    
    # Calculate validation metrics
    mse_val = mean_squared_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    mae_val = mean_absolute_error(y_val, y_pred_val)
    evs_val = explained_variance_score(y_val, y_pred_val)
    max_err_val = max_error(y_val, y_pred_val)
    medae_val = median_absolute_error(y_val, y_pred_val)
    
    # Create a table format for metrics
    print("Metric       |    Train     |  Validation")
    print("-------------|--------------|---------------")
    print(f"MSE          |   {mse_train:.2e}   |   {mse_val:.2e}")
    print(f"R²           |   {r2_train:.4f}     |   {r2_val:.4f}")
    print(f"MAE          |   {mae_train:.2e}   |   {mae_val:.2e}")
    print(f"EVS          |   {evs_train:.4f}     |   {evs_val:.4f}")
    print(f"Max Error    |   {max_err_train:.2e}   |   {max_err_val:.2e}")
    print(f"Median AE    |   {medae_train:.2e}   |   {medae_val:.2e}")
    print()
    
    # Return metrics in a dictionary for reporting
    return {
        "mse_train": mse_train,
        "r2_train": r2_train,
        "mse_val": mse_val,
        "r2_val": r2_val,
        "mae_train": mae_train,
        "evs_train": evs_train,
        "max_err_train": max_err_train,
        "medae_train": medae_train,
        "mae_val": mae_val,
        "evs_val": evs_val,
        "max_err_val": max_err_val,
        "medae_val": medae_val
    }

In [13]:
# Calculate metrics for Linear Regression model
lr_metrics = calculate_metrics(
    y_train, y_pred_train_lr, 
    y_val, y_pred_val_lr,
)

lr_metrics['name'] = "Linear Regression"
lr_metrics["model"] = lr
lr_metrics["params"] = None  # No params for Linear Regression


# Add to reports list
reports.append(lr_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.88e+13   |   9.11e+13
R²           |   0.8641     |   0.7986
MAE          |   4.45e+06   |   4.77e+06
EVS          |   0.8641     |   0.7988
Max Error    |   2.31e+08   |   2.86e+08
Median AE    |   2.76e+06   |   2.97e+06



## Random Forest

In [14]:
# Definir los hiperparámetros en un diccionario
rf_params = {
    'n_estimators': 100,       # cantidad de árboles
    'max_depth': None,         # sin profundidad máxima (puede sobreajustar si no se regula)
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42,
    'n_jobs': -1               # usar todos los núcleos disponibles
}

In [15]:
# Crear el modelo Random Forest
rf_reg = RandomForestRegressor(**rf_params)
rf_reg.fit(X_train, y_train)

# Predecir en el set de validación
y_pred_train_rf = rf_reg.predict(X_train)
y_pred_val_rf = rf_reg.predict(X_val)


In [16]:
# Calcular métricas para el modelo Random Forest
rf_metrics = calculate_metrics(
    y_train, y_pred_train_rf,
    y_val, y_pred_val_rf,
)

rf_metrics["name"] = "Random Forest"
rf_metrics["model"] = rf_reg
rf_metrics["params"] = rf_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(rf_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.30e+13   |   5.63e+13
R²           |   0.9744     |   0.8756
MAE          |   1.19e+06   |   2.89e+06
EVS          |   0.9744     |   0.8756
Max Error    |   2.14e+08   |   2.75e+08
Median AE    |   5.51e+05   |   1.39e+06



## XGBoost

In [17]:
# Definir los parámetros en un diccionario
xgb_params = {
    'objective': 'reg:squarederror',  # objetivo para regresión
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'random_state': 42
}

In [18]:
# Crear el modelo regresor con los parámetros
xgb_reg = XGBRegressor(**xgb_params)
xgb_reg.fit(X_train, y_train)

y_pred_train_xgb = xgb_reg.predict(X_train)
y_pred_val_xgb = xgb_reg.predict(X_val)


In [19]:
# Calcular métricas para el modelo XGBoost
xgb_metrics = calculate_metrics(
    y_train, y_pred_train_xgb,
    y_val, y_pred_val_xgb
)

xgb_metrics["name"] = "XGBoost"
xgb_metrics["model"] = xgb_reg
xgb_metrics["params"] = xgb_params  # Agregar los parámetros del modelo al reporte
# Agregar a la lista de reportes
reports.append(xgb_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   2.87e+13   |   5.10e+13
R²           |   0.9433     |   0.8871
MAE          |   2.90e+06   |   3.31e+06
EVS          |   0.9433     |   0.8872
Max Error    |   2.21e+08   |   2.24e+08
Median AE    |   1.81e+06   |   1.83e+06



## Red neuronal

In [20]:
# Definir los hiperparámetros en un diccionario
mlp_params = {
    'hidden_layer_sizes': (100, 50),   # dos capas ocultas con 100 y 50 neuronas
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,                   # L2 regularization
    'learning_rate': 'adaptive',
    'max_iter': 500,
    'random_state': 42
}

In [21]:
# Crear el modelo MLP
mlp_reg = MLPRegressor(**mlp_params)
mlp_reg.fit(X_train, y_train)

y_pred_train_mlp = mlp_reg.predict(X_train)
y_pred_val_mlp = mlp_reg.predict(X_val)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [22]:
# Calcular métricas para el modelo MLP
mlp_params_metrics = calculate_metrics(
    y_train, y_pred_train_mlp,
    y_val, y_pred_val_mlp
)

mlp_params_metrics["name"] = "MLP"
mlp_params_metrics["model"] = mlp_reg
mlp_params_metrics["params"] = mlp_params  # Agregar los parámetros del modelo al reporte

# Agregar a la lista de reportes
reports.append(mlp_params_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   8.19e+13   |   8.89e+13
R²           |   0.8382     |   0.8034
MAE          |   4.95e+06   |   5.07e+06
EVS          |   0.8382     |   0.8035
Max Error    |   3.02e+08   |   2.79e+08
Median AE    |   3.34e+06   |   3.40e+06



In [24]:
input_example = X_train.iloc[0:1]

for report in reports:
    with mlflow.start_run(run_name=report["name"]):
        if report["params"]:
            mlflow.log_params(report["params"])
        mlflow.log_metrics({
            "mse_val": report["mse_val"],
            "r2_val": report["r2_val"],
            "mae_val": report["mae_val"],
            "evs_val": report["evs_val"],
            "max_err_val": report["max_err_val"],
            "medae_val": report["medae_val"]
        })
        signature = infer_signature(input_example, report["model"].predict(input_example))
        if report["name"] == "XGBoost":
            mlflow.xgboost.log_model(report["model"], "model", input_example=input_example, signature=signature,  metadata={"Data Source": "Clean Train Data"})
        else:
            mlflow.sklearn.log_model(report["model"], "model", input_example=input_example, signature=signature, metadata={"Data Source": "Clean Train Data"})

2025/06/28 19:24:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 19:24:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: http://localhost:5000/#/experiments/306040637862002389/runs/dd4c3d0868464c6f9545e13016ed4894
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


2025/06/28 19:24:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/306040637862002389/runs/8f20b112a0404e60851e008fbbdd3203
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


/home/elianaostro/miniconda3/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [19:24:52] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/06/28 19:24:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/306040637862002389/runs/8a2458a502834058aa95ccb044ca8a09
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
🏃 View run MLP at: http://localhost:5000/#/experiments/306040637862002389/runs/51f78a1a69a0485cac848c42f018210c
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
